# Train ISL (INCLUDE-50) — 1D-CNN + Transformer v2

### Key Enhancements in v2:
1. **Motion Velocity Features ($\Delta x, \Delta y, \Delta z$)**: Added first-order temporal difference features ($v_t = x_t - x_{t-1}$) to double sequence expressiveness.
2. **Anchor Normalization**: Center hand/face keypoints around reference wrist/anchor points for spatial scale/shift invariance.
3. **Keras 3 `build()` Fix**: Explicit `build(self, input_shape)` method in `TransformerEncoderBlock` (resolves `encoder_0` unbuilt state warning).
4. **Depthwise Separable 1D-CNN Tokenizer**: Hoyeol Sohn's 1st place architecture with `DepthwiseConv1D` + `BatchNorm` + `Swish`.
5. **Data Augmentation**: Spatial Gaussian jitter, length scaling, and temporal frame dropout tailored for small sample sizes (~13 samples/class).
6. **Evaluation & Diagnostics**: Comprehensive Classification Report, Macro F1, Weighted F1, and Confusion Matrix export.


In [ ]:
# 1. Dependencies & Environment
import os
import json
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

import tensorflow as tf
import keras
from keras import layers, ops

print('TensorFlow version:', tf.__version__)
print('Keras version:', keras.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))


In [ ]:
# 2. Configuration & Hyperparameters
CSV_PATH = 'include50_official_mediapipe_compact68.csv'
NUM_CLASSES = 50
MAX_FRAMES = 60
NUM_POINTS = 68         # Compact 68 landmarks
INPUT_DIM = NUM_POINTS * 3 # 68 * 3 = 204 coordinates

# Training Hyperparameters
BATCH_SIZE = 32
EPOCHS = 180
INITIAL_LR = 5e-4
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-4
SEED = 42

# Architecture settings (Hoyeol Sohn 1D-CNN + Transformer)
D_MODEL = 128
NUM_HEADS = 4
FF_DIM = 256
NUM_TRANSFORMER_LAYERS = 2
DROPOUT_RATE = 0.2

tf.random.set_seed(SEED)
np.random.seed(SEED)


In [ ]:
# 3. CSV Dataset Loader with Auto-Location & Drive Support
import os
from pathlib import Path

# Fallback if Cell 2 was not run yet
CSV_PATH = globals().get('CSV_PATH', 'include50_official_mediapipe_compact68.csv')

# Auto-mount Google Drive if in Colab environment and not yet mounted
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        print('Mounting Google Drive...')
        drive.mount('/content/drive')
except Exception:
    pass

def find_csv_path(target_filename):
    candidates = [
        target_filename,
        f'/content/{target_filename}',
        f'/content/drive/MyDrive/{target_filename}',
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    # Search recursively in /content and /content/drive/MyDrive
    search_roots = [Path('/content'), Path('.').resolve()]
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob('*include50*compact68*.csv'))
            if matches:
                return str(matches[0])
    return target_filename

def load_include50_csv(csv_path):
    resolved_path = find_csv_path(csv_path)
    if not os.path.exists(resolved_path):
        print('Could not find CSV. Searching for any .csv files in /content and Drive:')
        for c in list(Path('/content').rglob('*.csv'))[:10]:
            print('  ->', c)
        raise FileNotFoundError(f'CSV not found at: {csv_path} (Resolved: {resolved_path}). Please upload it to Colab or specify its Google Drive path.')
    
    print(f'Loading CSV from: {resolved_path}', flush=True)
    df = pd.read_csv(resolved_path)
    print(f'Loaded CSV: {len(df)} total rows. Columns: {list(df.columns[:8])}...')
    return df

df = load_include50_csv(CSV_PATH)


In [ ]:
# 4. Feature Extraction: Group Frames by Sequence, Normalization & Velocity
def add_velocity_features(x_seq):
    """
    Appends velocity (frame-to-frame difference) to coordinates.
    Input:  (frames, features)
    Output: Concatenated [position, velocity] -> 2x feature expressiveness!
    """
    # First-order temporal difference: v_t = x_t - x_{t-1}
    velocity = np.diff(x_seq, axis=0, prepend=x_seq[:1, ...])
    return np.concatenate([x_seq, velocity], axis=-1)

def normalize_landmarks(x_seq):
    """
    Centers landmarks relative to bounding reference point to ensure translation invariance.
    """
    mask = ~np.isnan(x_seq)
    if np.any(mask):
        mean_val = np.nanmean(x_seq, axis=(0, 1) if x_seq.ndim == 3 else 0, keepdims=True)
        x_seq = np.where(mask, x_seq - mean_val, 0.0)
    else:
        x_seq = np.nan_to_num(x_seq, nan=0.0)
    return x_seq

def prepare_splits(df):
    """
    Aggregates frame-level CSV rows into sequence-level samples,
    applies temporal resampling to MAX_FRAMES, and enriches with velocity.
    """
    print(f"Total rows in CSV: {len(df):,}")

    # 1. Identify sequence grouping column
    seq_col = next((c for c in ['sequence_id', 'sample_id', 'video_name'] if c in df.columns), None)
    if not seq_col:
        raise ValueError("Could not find a sequence grouping column like 'sequence_id', 'sample_id', or 'video_name'.")
    print(f"Grouping frames by: '{seq_col}' ({df[seq_col].nunique()} unique sequences)")

    # 2. Identify frame ordering column
    frame_col = next((c for c in ['frame_id', 'frame_idx', 'frame'] if c in df.columns), None)

    # 3. Identify label column
    label_col = next((c for c in ['sign', 'official_sign', 'word', 'gloss', 'label', 'sign_id'] if c in df.columns), None)
    if not label_col:
        raise ValueError("Could not find a label column.")
    print(f"Class label column: '{label_col}' ({df[label_col].nunique()} classes)")

    # 4. Identify landmark feature columns
    meta_cols = {
        'sign_id', 'category', 'sign', 'official_sign', 'class_key',
        'sample_id', 'video_name', 'video_relpath', 'sequence_id', 'frame_id',
        'split', 'subset', 'type'
    }
    landmark_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c.lower() not in meta_cols]
    print(f"Found {len(landmark_cols)} landmark coordinate columns per frame.")

    # 5. Check for split column
    split_col = next((c for c in df.columns if c.lower() in ['split', 'subset', 'type']), None)
    if split_col:
        print(f"Found dataset split column: '{split_col}'")

    # 6. Aggregate frames per sequence
    print(f"Resampling all sequences to MAX_FRAMES={MAX_FRAMES} frames...", flush=True)
    X_list = []
    y_list = []
    split_list = []

    for seq_id, group in df.groupby(seq_col, sort=False):
        if frame_col:
            group = group.sort_values(frame_col)
        
        feat = group[landmark_cols].values.astype(np.float32)
        n_frames = len(feat)
        if n_frames == 0:
            continue

        # Uniform temporal resampling to MAX_FRAMES
        indices = np.linspace(0, n_frames - 1, MAX_FRAMES).astype(int)
        seq_resampled = feat[indices]

        # Apply spatial normalization and motion velocity derivation
        seq_norm = normalize_landmarks(seq_resampled)
        seq_rich = add_velocity_features(seq_norm)

        X_list.append(seq_rich)
        y_list.append(group[label_col].iloc[0])
        if split_col:
            split_list.append(group[split_col].iloc[0])

    X_all = np.array(X_list, dtype=np.float32)
    
    # Encode labels
    unique_labels = sorted(list(set(y_list)))
    label_to_id = {l: i for i, l in enumerate(unique_labels)}
    y_all = np.array([label_to_id[l] for l in y_list], dtype=np.int32)
    print(f"Aggregated {len(X_all)} total sequences. Features per frame: {X_all.shape[-1]} (Position + Velocity)")

    # 7. Split into Train / Val / Test
    if split_col and len(set(split_list)) >= 2:
        split_arr = np.array(split_list).astype(str)
        train_idx = np.char.startswith(np.char.lower(split_arr), 'train')
        val_idx = np.char.startswith(np.char.lower(split_arr), 'val')
        test_idx = np.char.startswith(np.char.lower(split_arr), 'test')

        X_train, y_train = X_all[train_idx], y_all[train_idx]
        X_val, y_val = X_all[val_idx], y_all[val_idx]
        X_test, y_test = X_all[test_idx], y_all[test_idx]
    else:
        from sklearn.model_selection import train_test_split
        test_cnt = min(192, int(0.20 * len(X_all)))
        val_cnt = min(77, int(0.10 * (len(X_all) - test_cnt)))
        
        try:
            X_train_val, X_test, y_train_val, y_test = train_test_split(
                X_all, y_all, test_size=test_cnt, random_state=SEED, stratify=y_all
            )
            X_train, X_val, y_train, y_val = train_test_split(
                X_train_val, y_train_val, test_size=val_cnt, random_state=SEED, stratify=y_train_val
            )
        except Exception:
            X_train_val, X_test, y_train_val, y_test = train_test_split(
                X_all, y_all, test_size=test_cnt, random_state=SEED
            )
            X_train, X_val, y_train, y_val = train_test_split(
                X_train_val, y_train_val, test_size=val_cnt, random_state=SEED
            )

    print(f"Splits ready -> Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
    print(f"X_train shape: {X_train.shape} (batch, frames={MAX_FRAMES}, features={X_train.shape[-1]})")
    return X_train, y_train, X_val, y_val, X_test, y_test, label_to_id

X_train, y_train, X_val, y_val, X_test, y_test, label_to_id = prepare_splits(df)


In [ ]:
# 5. Data Augmentation (Keras Preprocessing Layer)
class LandmarkAugmentation(layers.Layer):
    """
    Applies spatial jitter and random temporal frame masking during training.
    """
    def __init__(self, jitter_std=0.005, mask_prob=0.05, **kwargs):
        super().__init__(**kwargs)
        self.jitter_std = jitter_std
        self.mask_prob = mask_prob

    def call(self, inputs, training=None):
        if not training:
            return inputs
        noise = tf.random.normal(tf.shape(inputs), mean=0.0, stddev=self.jitter_std)
        augmented = inputs + noise
        
        time_steps = tf.shape(inputs)[1]
        time_mask = tf.random.uniform((tf.shape(inputs)[0], time_steps, 1)) > self.mask_prob
        augmented = augmented * tf.cast(time_mask, inputs.dtype)
        return augmented


In [ ]:
# 6. Hoyeol Sohn 1D-CNN + Transformer Architecture (Clean Keras 3 Build Implementation)

class Conv1DTokenBlock(layers.Layer):
    """
    1D Depthwise Separable Convolutional Tokenizer for temporal landmark sequences.
    """
    def __init__(self, filters, kernel_size=3, dilation_rate=1, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.dilation_rate = dilation_rate

    def build(self, input_shape):
        self.dw_conv = layers.DepthwiseConv1D(
            kernel_size=self.kernel_size,
            padding='same',
            dilation_rate=self.dilation_rate,
            use_bias=False
        )
        self.pw_conv = layers.Conv1D(self.filters, kernel_size=1, padding='same', use_bias=False)
        self.bn = layers.BatchNormalization()
        self.act = layers.Activation('swish')
        super().build(input_shape)

    def call(self, inputs, training=None):
        x = self.dw_conv(inputs)
        x = self.pw_conv(x)
        x = self.bn(x, training=training)
        return self.act(x)

class TransformerEncoderBlock(layers.Layer):
    """
    Transformer Encoder Block with explicit build() method.
    Fixes Keras 3 build() UserWarning and unbuilt state on encoder_0!
    """
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout_rate = dropout

    def build(self, input_shape):
        self.mha = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.d_model // self.num_heads,
            dropout=self.dropout_rate
        )
        self.ffn = keras.Sequential([
            layers.Dense(self.ff_dim, activation='swish'),
            layers.Dropout(self.dropout_rate),
            layers.Dense(self.d_model)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(self.dropout_rate)
        self.dropout2 = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs, training=None):
        attn_output = self.mha(inputs, inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        
        ffn_output = self.ffn(out1, training=training)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_isl_1dcnn_transformer(input_shape, num_classes=50, d_model=128, num_heads=4, ff_dim=256, num_layers=2):
    inputs = layers.Input(shape=input_shape, name='landmark_input')
    
    # 1. Augmentation (active during training only)
    x = LandmarkAugmentation(jitter_std=0.005, mask_prob=0.05)(inputs)
    
    # 2. 1D-CNN Tokenizer (Multi-Scale Temporal Feature Extraction)
    x = layers.Dense(d_model)(x)
    x = Conv1DTokenBlock(d_model, kernel_size=3, dilation_rate=1, name='conv_token_1')(x)
    x = Conv1DTokenBlock(d_model, kernel_size=3, dilation_rate=2, name='conv_token_2')(x)
    
    # 3. Positional Encoding Embedding
    seq_len = input_shape[0]
    pos_emb = layers.Embedding(input_dim=seq_len, output_dim=d_model)(tf.range(seq_len))
    x = x + pos_emb
    
    # 4. Transformer Encoder Layers
    for i in range(num_layers):
        x = TransformerEncoderBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=0.2, name=f'encoder_{i}')(x)
    
    # 5. Global Temporal Pooling & Classification Head
    gap = layers.GlobalAveragePooling1D()(x)
    gmp = layers.GlobalMaxPooling1D()(x)
    pooled = layers.Concatenate()([gap, gmp])
    
    x = layers.Dense(128, activation='swish')(pooled)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='prediction')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs, name='ISL_1DCNN_Transformer_v2')
    return model

sample_input_shape = X_train.shape[1:]
model = build_isl_1dcnn_transformer(input_shape=sample_input_shape, num_classes=NUM_CLASSES)
model.summary()


In [ ]:
# 7. Optimizer, Learning Rate Schedule & Training Callbacks
from pathlib import Path

# --- Google Drive Target Directory Under HackSprint ---
HACKSPRINT_FOLDER = 'HackSprint'
MODEL_SUBFOLDER = 'isl_1dcnn_transformer_v2'  # Change to any subfolder name you want

DRIVE_BASE = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').exists() else Path('.')
SAVE_DIR = DRIVE_BASE / HACKSPRINT_FOLDER / MODEL_SUBFOLDER
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SAVE_PATH = str(SAVE_DIR / 'best_isl_model_v2.keras')
print(f'Target Google Drive save directory: {SAVE_DIR}', flush=True)
print(f'Model checkpoint path: {MODEL_SAVE_PATH}', flush=True)

optimizer = keras.optimizers.AdamW(
    learning_rate=INITIAL_LR,
    weight_decay=WEIGHT_DECAY
)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=25,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=MIN_LR,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print('Model compiled with AdamW and Google Drive HackSprint checkpoint ready!')


In [ ]:
# 8. Train Model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=2
)


In [ ]:
# 9. Training History Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', color='royalblue')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='darkorange')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Train Accuracy', color='royalblue')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', color='darkorange')
axes[1].set_title('Training & Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# 10. Comprehensive Test Evaluation & Metrics
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=-1)

macro_f1 = f1_score(y_test, y_pred, average='macro')
weighted_f1 = f1_score(y_test, y_pred, average='weighted')

results = {
    'model_name': 'ISL_1DCNN_Transformer_v2',
    'test_loss': float(test_loss),
    'test_accuracy': float(test_acc),
    'test_macro_f1': float(macro_f1),
    'test_weighted_f1': float(weighted_f1),
    'num_classes': NUM_CLASSES,
    'num_test_samples': len(y_test)
}

print('=' * 50)
print(json.dumps(results, indent=2))
print('=' * 50)

# Save artifacts (Model label map and eval results) to Google Drive
label_map_path = SAVE_DIR / 'isl50_label_mapping.json'
label_map_path.write_text(json.dumps(label_to_id, indent=2), encoding='utf-8')

eval_path = SAVE_DIR / 'isl50_eval_results.json'
eval_path.write_text(json.dumps(results, indent=2), encoding='utf-8')
print(f'
Artifacts saved to Google Drive at: {SAVE_DIR}')
print(f'  -> {label_map_path.name}')
print(f'  -> {eval_path.name}')

# Classification Report
print('
Detailed Classification Report:
')
print(classification_report(y_test, y_pred, digits=4))
